# Basics &mdash; Function Signatures: Domain, Codomain, Range, Onto, Total

**Concept 12 of the Basics decomposition:** *Function Signatures: Domain, Codomain, Range, Onto/Into, Total*

The vocabulary for reading every transition-function signature in the book, in one place.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Basics/Concept-Function-Signatures/Concept-Function-Signatures.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Basics/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.LangDef        import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


For $f : A \to B$: $A$ is the **domain**, $B$ the **codomain**, and
$$rng(f) = \{y \in B : \exists x \in A,\ f(x)=y\}$$
is the **range** &mdash; what $f$ actually hits. $f$ is **onto** if $rng(f)=B$, **into**
otherwise; **total** if defined everywhere on $A$.

**All functions are assumed total by default**, but one may specify partially and then
**totalize** &mdash; which is exactly Chapter 4's black-hole construction.

The appendix then does something unusually useful: it lists **every machine signature
in the book** together.

| machine | signature |
|---|---|
| DFA | $\delta : Q\times\Sigma \to Q$ |
| NFA | $\delta : Q\times\Sigma_\varepsilon \to {\cal P}(Q)$ |
| PDA | $\Delta : Q\times(\Sigma\cup\{\varepsilon\})\times(\Gamma\cup\{\varepsilon\}) \to {\cal P}(Q\times\Gamma^*)$ |
| DTM | $\Delta : Q\times\Gamma \to Q\times\Gamma\times\{L,R,S\}$ |
| NDTM | the same, with ${\cal P}(\cdots)$ |

Read down that column and the book's architecture is visible: **${\cal P}$ marks
nondeterminism, extra domain components mark extra memory.**

## 2. Definitions

### Domain, codomain, range, onto, total

In [ ]:
def rng(f, A):      return {f(a) for a in A}
def is_onto(f, A, B):  return rng(f, A) == set(B)
def is_into(f, A, B):  return rng(f, A) < set(B)
def describe(name, f, A, B):
    r = rng(f, A)
    print("  %-14s range %-18s onto? %-6s into? %s"
          % (name, sorted(r), is_onto(f, A, B), is_into(f, A, B)))

### The signature table, as data

In [ ]:
SIGS = [
 ("DFA",  "delta : Q x Sigma -> Q",                                   "deterministic, finite memory"),
 ("NFA",  "delta : Q x Sigma_eps -> P(Q)",                            "P => nondeterminism"),
 ("PDA",  "Delta : Q x (Sigma+eps) x (Gamma+eps) -> P(Q x Gamma*)",   "+ a stack"),
 ("DTM",  "Delta : Q x Gamma -> Q x Gamma x {L,R,S}",                 "+ a writable tape"),
 ("NDTM", "Delta : Q x Gamma -> P(Q x Gamma x {L,R,S})",              "P again"),
]

<!-- nav-strip -->

---

&larr;&nbsp;[Basics&nbsp;11.&nbsp;Cartesian Product, Binary Relations, and Functions](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Basics/Concept-Cartesian-Product-Relations-Functions/Concept-Cartesian-Product-Relations-Functions.ipynb) &nbsp;&middot;&nbsp; [**Basics** index](https://github.com/ganeshutah/Jove/blob/master/Basics/README.md) &nbsp;&middot;&nbsp; [Basics&nbsp;13.&nbsp;Trees: Root, Height, Branching Factor, and the $b^n$ Bound](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Basics/Concept-Trees-And-The-Branching-Bound/Concept-Trees-And-The-Branching-Bound.ipynb)&nbsp;&rarr;

---

## 3. Tests

Range, onto and into.

In [ ]:
A, B = set(range(6)), set(range(6))
describe("x -> x",      lambda x: x,          A, B)
describe("x -> x % 3",  lambda x: x % 3,      A, B)
describe("x -> 0",      lambda x: 0,          A, B)
assert is_onto(lambda x: x, A, B)
assert is_into(lambda x: x % 3, A, B)

**Total by default**, and totalizing a partial specification.

In [ ]:
partial = {0: 'a', 2: 'b'}
A = set(range(4))
print("  partial map :", partial, " total on", sorted(A), "?",
      set(partial) == A)
assert set(partial) != A
total = dict(partial)
for x in A:
    total.setdefault(x, 'BH')          # a default at every undefined point
print("  totalized   :", total)
assert set(total) == A
print("\nThat default IS the black hole of Chapter 4, Concept 5.")

A DFA's $\delta$, read through the vocabulary.

In [ ]:
D = md2mc('''DFA
IF : 0 -> Od
IF : 1 -> IF
Od : 0 -> IF
Od : 1 -> Od
''')
dom = {(q, a) for q in D["Q"] for a in D["Sigma"]}
codom = D["Q"]
r = set(D["Delta"].values())
print("  domain   Q x Sigma :", sorted(dom))
print("  codomain Q         :", sorted(codom))
print("  range              :", sorted(r))
print("  total?", set(D["Delta"]) == dom, "   onto?", r == codom)
assert set(D["Delta"]) == dom

A machine whose $\delta$ is **into**, not onto: a state nothing reaches.

In [ ]:
# 'Un' must not be the TARGET of any transition, or it would be in the
# range after all -- a self-loop would put it there.
E = md2mc('''DFA
I  : 0 -> I
I  : 1 -> F
F  : 0 | 1 -> F
Un : 0 | 1 -> I
''')
r = set(E["Delta"].values())
print("  Q     :", sorted(E["Q"]))
print("  range :", sorted(r))
print("  onto? ", r == E["Q"], "  into? ", r < E["Q"])
assert r < E["Q"]
print("\n'Un' is in the codomain but not the range -- delta is INTO.")
print("(It is also unreachable, which is what pruneUnreach removes.)")

**The signature table &mdash; the whole book in five lines.**

In [ ]:
print("%-6s %-56s %s" % ("machine", "signature", "what changed"))
for m, s, why in SIGS:
    print("%-6s %-56s %s" % (m, s, why))
print()
print("P(...)                 -> nondeterminism")
print("an extra domain factor -> extra memory (a stack, a tape)")
print("a codomain with Gamma* -> the machine may WRITE more than it read")

## 4. Animation

A DFA whose $\delta$ is total on $Q\times\Sigma$ and onto $Q$.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(md2mc('DFA\nIF : 0 -> Od\nIF : 1 -> IF\nOd : 0 -> IF\nOd : 1 -> Od\n'), FuseEdges=True)

## 5. Exercises


1. Is a DFA's $\delta$ ever onto? Ever not? Give one of each.
2. Write the LBA signature. How does it differ from the TM's?
3. Which signature in the table has the largest codomain, and why?

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 252 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Basics/Concept-Function-Signatures')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')